# Gemma Remember: A Gentle AI Companion for Dementia Care

**Gemma 4 Good Hackathon Submission**

---

### The Problem

Over 55 million people worldwide live with dementia. One of the most painful symptoms is forgetting the faces and voices of loved ones. Caregivers repeat the same introductions dozens of times a day. It's exhausting for them and distressing for the person with dementia.

### Our Solution

**Gemma Remember** is a fully offline, privacy-first app that uses **Gemma 4** to help someone with dementia remember their family. Caregivers upload family photos, voice clips, and short stories. When the person shows a photo or asks "Who is this?", the app responds warmly and personally — grounded entirely in real family data. No hallucinations. No cloud. No data ever leaves the device.

### How It Works

1. **Data**: Family photos + captions + voice clips → converted to Gemma 4 multimodal chat format
2. **Fine-tune**: LoRA fine-tuning with Unsloth on ~200-500 personal examples (runs on consumer GPU)
3. **Deploy**: Export to GGUF → runs on a tablet or laptop with no internet
4. **Use**: Show a photo → get a warm, personal reminder

### Why Gemma 4?

- **Multimodal**: Understands images natively — no separate vision pipeline needed
- **Small enough**: E2B fits in 8GB VRAM, E4B in 16GB — runs on real consumer hardware
- **Open**: Can be fine-tuned and deployed fully offline — critical for privacy
- **GGUF export**: Runs on Android tablets via llama.cpp, no internet required

---

## Step 0: Setup

Install dependencies. On Kaggle, select a **GPU T4 x2** or **P100** runtime.

In [ ]:
%%capture
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes xformers
!pip install Pillow pyyaml librosa soundfile

## Step 1: Create Mock Family Data

For this demo we generate synthetic family data. In real use, caregivers upload actual photos and write captions like: *"This is your daughter Sarah. She loved baking cookies with you every Sunday."*

In [ ]:
import json
import os
import re
from pathlib import Path
from PIL import Image
import random

MOCK_DIR = "/tmp/gemma_remember_mock"
PHOTOS_DIR = f"{MOCK_DIR}/photos"
CAPTIONS_DIR = f"{MOCK_DIR}/captions"
PROCESSED_DIR = f"{MOCK_DIR}/processed"

for d in [PHOTOS_DIR, CAPTIONS_DIR, PROCESSED_DIR]:
    os.makedirs(d, exist_ok=True)

# Our mock family — each entry is a memory anchor point
FAMILY = [
    ("sarah_birthday", "This is your daughter Sarah. She turned 35 last March. "
     "She brought that chocolate cake you love — the one with the raspberries on top. "
     "She lives in Portland now but calls you every Sunday."),
    ("arki_birdhouse", "This is your son Arki. He built you that birdhouse in the "
     "backyard in 1998 — remember, the blue one with the crooked roof? He was so "
     "proud of it. He's an engineer now, just like you always said he'd be."),
    ("maria_wedding", "This is your wife Maria on your wedding day, June 14th, 1975. "
     "She wore her mother's dress. You danced to 'Unforgettable' by Nat King Cole. "
     "She said it was the happiest day of her life."),
    ("tommy_graduation", "This is your grandson Tommy at his high school graduation. "
     "Class of 2023. He got a scholarship to study marine biology — he loves the ocean "
     "just like you do. He still wears the watch you gave him."),
    ("bella_dog", "This is Bella, your golden retriever. She's been with you for 8 years. "
     "She loves belly rubs and sleeps at the foot of your bed every night. "
     "Sarah got her for you after you retired."),
    ("family_christmas_2022", "This is Christmas 2022 at your house. Everyone came — "
     "Sarah, Arki, Tommy, little Maya, and Maria made her famous tamales. "
     "Arki played guitar and you all sang carols. Maya sat on your lap the whole time."),
    ("maya_first_steps", "This is your great-granddaughter Maya taking her first steps. "
     "She was 11 months old. She walked straight to you — you were sitting in your "
     "favorite chair. Everyone cheered. You cried happy tears."),
    ("fishing_trip_1995", "This is you and Arki on your fishing trip to Lake Tahoe in 1995. "
     "You caught a 12-pound trout and Arki caught nothing — he still jokes about it. "
     "You both got sunburned and ate hot dogs for dinner."),
]

# Create placeholder images (colored squares — in real use these are family photos)
for stem, caption in FAMILY:
    random.seed(stem)
    color = tuple(random.randint(80, 220) for _ in range(3))
    img = Image.new("RGB", (224, 224), color)
    img.save(f"{PHOTOS_DIR}/{stem}.jpg")
    Path(f"{CAPTIONS_DIR}/{stem}.txt").write_text(caption)

print(f"Created {len(FAMILY)} mock family memories")
print(f"Photos: {PHOTOS_DIR}")
print(f"Captions: {CAPTIONS_DIR}")

## Step 2: Convert to Gemma 4 Chat Format

Each family memory becomes multiple training examples:
- Image + "Who is this?" → warm reminder (primary)
- Image + variant questions → same reminder (robustness)
- Text-only "Tell me about Sarah" → reminder without photo

This teaches the model to respond to however the person asks.

In [ ]:
SYSTEM_PROMPT = """You are a gentle memory companion. You help someone with dementia
remember their loved ones. You ONLY use information from the family
memories you've been given. Never invent details. Be warm, patient,
and reassuring. If you're unsure, say "I'm not sure, but let's look
at your photos together."""

QUESTIONS = [
    "Who is this person?",
    "Who is this?",
    "Do I know them?",
    "Tell me about this photo.",
    "Can you help me remember?",
    "I think I know this person... who are they?",
    "This face looks familiar. Who is it?",
]

def build_vision_example(photo_path, caption, question):
    """One image + question → warm reminder."""
    return {"messages": [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [
            {"type": "image", "image": photo_path},
            {"type": "text", "text": question},
        ]},
        {"role": "model", "content": [{"type": "text", "text": caption}]},
    ]}

def build_text_example(stem, caption):
    """Text-only: ask by name."""
    name = re.sub(r'[_\d]+', ' ', stem).strip().title()
    q = random.choice([f"Tell me about {name}.", f"Who is {name}?", f"What do you know about {name}?"])
    return {"messages": [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [{"type": "text", "text": q}]},
        {"role": "model", "content": [{"type": "text", "text": caption}]},
    ]}

# Build dataset
examples = []
for stem, caption in FAMILY:
    photo_path = f"{PHOTOS_DIR}/{stem}.jpg"
    # 3 image+question variants
    random.seed(stem)
    chosen_qs = random.sample(QUESTIONS, 3)
    for q in chosen_qs:
        examples.append(build_vision_example(photo_path, caption, q))
    # 1 text-only variant
    examples.append(build_text_example(stem, caption))

# Save JSONL
train_path = f"{PROCESSED_DIR}/train.jsonl"
with open(train_path, "w") as f:
    for ex in examples:
        f.write(json.dumps(ex) + "\n")

print(f"Created {len(examples)} training examples from {len(FAMILY)} family memories")
print(f"  {len(FAMILY)*3} image+question pairs + {len(FAMILY)} text-only = {len(examples)} total")
print(f"Saved to {train_path}")

## Step 3: Load Gemma 4 with Unsloth

We use **Gemma 4 E4B** (4 billion params, multimodal) with LoRA — only ~2% of parameters are trainable. Unsloth gives us 1.5x speed and 50% less VRAM.

In [ ]:
from unsloth import FastVisionModel

model, tokenizer = FastVisionModel.from_pretrained(
    "google/gemma-4-e4b-it",
    load_in_16bit=True,
    max_seq_length=2048,
)

print("Base model loaded.")

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_rslora=True,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

## Step 4: Prepare Dataset for Training

Load the JSONL and convert image paths to actual PIL images.

In [ ]:
from datasets import load_dataset

def convert_example(example):
    """Load images from disk into PIL objects for the trainer."""
    converted = []
    for msg in example["messages"]:
        new_content = []
        for part in msg["content"]:
            if part["type"] == "image":
                try:
                    img = Image.open(part["image"]).convert("RGB")
                    new_content.append({"type": "image", "image": img})
                except Exception as e:
                    print(f"  Warning: {e}")
            else:
                new_content.append(part)
        converted.append({"role": msg["role"], "content": new_content})
    return {"messages": converted}

dataset = load_dataset("json", data_files=train_path, split="train")
dataset = dataset.map(convert_example, remove_columns=dataset.column_names)
print(f"Dataset ready: {len(dataset)} examples")

## Step 5: Fine-Tune

Conservative settings for consumer hardware:
- Batch size 1, gradient accumulation 4 (effective batch = 4)
- Gradient checkpointing to save VRAM
- 3 epochs over our small dataset
- No cloud logging — `report_to="none"`

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.trainer import UnslothVisionDataCollator

OUTPUT_DIR = "/tmp/gemma_remember_model"

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="linear",
    warmup_steps=5,
    logging_steps=5,
    save_steps=50,
    seed=42,
    fp16=True,
    gradient_checkpointing=True,
    optim="adamw_8bit",
    report_to="none",
    remove_unused_columns=False,
    dataset_text_field="",
    dataset_kwargs={"skip_prepare_dataset": True},
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=sft_config,
    train_dataset=dataset,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
)

print("Starting training...")
trainer.train()
print("Training complete!")

In [ ]:
# Save the LoRA adapter
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

## Step 6: Test — "Who Is This?"

The moment that matters. Show a photo, ask a question, get a gentle reminder.

In [ ]:
FastVisionModel.for_inference(model)

def ask_gemma_remember(image_path=None, question="Who is this person?"):
    """Ask Gemma Remember about a photo or a person."""
    if image_path:
        img = Image.open(image_path).convert("RGB")
        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
            {"role": "user", "content": [
                {"type": "image", "image": img},
                {"type": "text", "text": question},
            ]},
        ]
    else:
        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
            {"role": "user", "content": [{"type": "text", "text": question}]},
        ]

    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt",
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.3,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.1,
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "<start_of_turn>model" in response:
        response = response.split("<start_of_turn>model")[-1].strip()
    if "<end_of_turn>" in response:
        response = response.split("<end_of_turn>")[0].strip()
    return response

print("Gemma Remember is ready. Let's remember together.")

In [ ]:
# Test 1: Show a photo — "Who is this?"
print("Showing photo of Sarah's birthday...")
print()
response = ask_gemma_remember(f"{PHOTOS_DIR}/sarah_birthday.jpg", "Who is this person?")
print(f"Gemma Remember: {response}")

In [ ]:
# Test 2: Show a different photo with a different question
print("Showing photo of the fishing trip...")
print()
response = ask_gemma_remember(f"{PHOTOS_DIR}/fishing_trip_1995.jpg", "I think I know this place... can you help me remember?")
print(f"Gemma Remember: {response}")

In [ ]:
# Test 3: Text-only — ask by name
print("Asking about Maria (no photo)...")
print()
response = ask_gemma_remember(question="Tell me about Maria.")
print(f"Gemma Remember: {response}")

In [ ]:
# Test 4: The safety check — asking about someone NOT in the data
print("Asking about an unknown person...")
print()
response = ask_gemma_remember(question="Who is Robert?")
print(f"Gemma Remember: {response}")
print()
print("(Expected: the model should say it's not sure, not hallucinate a 'Robert')")

## Step 7: Export for Offline Deployment

Export to GGUF so this runs on a tablet with no internet — just the person and their memories.

In [ ]:
GGUF_DIR = "/tmp/gemma_remember_gguf"

model.save_pretrained_gguf(
    GGUF_DIR,
    tokenizer,
    quantization_method="q4_k_m",  # ~4GB — fits on most tablets
)

print(f"Exported to {GGUF_DIR}/")
print("This file can run on:")
print("  - Ollama (desktop): ollama create gemma-remember -f Modelfile")
print("  - llama.cpp (desktop): ./llama-server -m model.gguf --port 8080")
print("  - Android tablet: via LiteRT or llama.cpp Android build")

## Impact

### Who This Helps

- **55 million people** living with dementia worldwide (WHO, 2023)
- **11 million unpaid caregivers** in the US alone (Alzheimer's Association)
- Families in **low-resource settings** where professional memory care is unaffordable

### Why It Matters

A person with dementia may not remember their daughter's name — but when shown a photo and gently told *"This is Sarah, she bakes cookies with you every Sunday"*, the emotional connection can resurface. Gemma Remember doesn't cure dementia. It anchors moments of recognition that bring comfort to both the person and their family.

### Privacy as a Feature, Not an Afterthought

Family photos and voice clips are the most intimate data a person has. Gemma Remember:
- Runs **100% offline** after setup
- Stores **nothing in the cloud**
- Fine-tunes on **the user's device**
- Can be exported to a **standalone GGUF file** — no internet ever

### Scalability

- Works on a **$300 Android tablet** with the GGUF export
- Caregivers can set it up in **under an hour** (upload photos, write captions)
- Fine-tuning takes **~30 minutes** on a consumer GPU
- The model is **personal to each family** — no generic responses

---

### Open Source

Full code: [github.com/Javierg720/gemma-remember](https://github.com/Javierg720/gemma-remember)

MIT License — use it, adapt it, help someone you love remember.